# BERT + LoRA — Multi-label Speech Act Classifier
**AIML339 Email Thread Triage**

This notebook trains a multi-label classifier on the BC3 corpus to predict speech act labels
(Request, Propose, Commit, Meeting, Subjective, Informative) for each email.

It uses `bert-base-uncased` as the base model, fine-tuned with LoRA (Low-Rank Adaptation)
so that only a small fraction of parameters are trained. The output layer uses sigmoid + BCE loss
to handle multi-label classification independently for each of the 6 labels.

Results are saved to `results/bc3_bert_lora_test_predictions.csv` in the same format as the
TF-IDF baseline, enabling a direct metric comparison.

**Run on Google Colab with GPU runtime** (Runtime > Change runtime type > T4 GPU).

## 0. Install dependencies

These libraries are not pre-installed in Colab by default.
- `transformers`: provides BERT and the tokenizer.
- `peft`: provides LoRA (Parameter-Efficient Fine-Tuning).
- `accelerate`: required internally by `transformers` for training utilities.

In [14]:
!pip install -q transformers peft accelerate
!pip install -q -U torchao

## 1. Configuration and data loading

All tunable parameters live here so they are easy to find and change without touching the rest of the notebook.

Key decisions:
- `MAX_LEN = 256`: BERT supports up to 512 tokens, but BC3 emails are short. 256 is enough and halves the GPU memory needed per email.
- `BATCH_SIZE = 8`: how many emails the model processes before updating its parameters. Safe for Colab's free T4 GPU.
- `LR = 2e-4`: LoRA allows a higher learning rate than full fine-tuning because only the small added matrices are being updated.
- `THRESHOLD = 0.5`: after the sigmoid, probabilities above this are predicted as active labels.

In [15]:
import pandas as pd
import numpy as np
import torch
import random
import os

# --- Paths -------------------------------------------------------------------
# If running in Colab, upload bc3_emails_labeled.csv or mount Google Drive
# and adjust DATA_PATH accordingly. Example for Drive mount:
#   DATA_PATH = "/content/drive/MyDrive/aiml339/data/bc3_emails_labeled.csv"
DATA_PATH   = "/content/bc3_emails_labeled.csv"  # works when running locally
RESULTS_DIR = "../results/"

# --- Model -------------------------------------------------------------------
MODEL_NAME = "bert-base-uncased"

# --- Labels ------------------------------------------------------------------
LABEL_COLS  = ["label_Request", "label_Propose", "label_Commit",
               "label_Meeting", "label_Subjective", "label_Informative"]
LABEL_NAMES = ["Request", "Propose", "Commit", "Meeting", "Subjective", "Informative"]
NUM_LABELS  = len(LABEL_NAMES)

# --- Hyperparameters ---------------------------------------------------------
MAX_LEN    = 256
BATCH_SIZE = 8
EPOCHS     = 5
LR         = 2e-4
THRESHOLD  = 0.5
SEED       = 42

# --- Reproducibility ---------------------------------------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Device ------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Load data ---------------------------------------------------------------
def load_splits(path):
    df = pd.read_csv(path)
    train = df[df["split"] == "train"].reset_index(drop=True)
    val   = df[df["split"] == "val"].reset_index(drop=True)
    test  = df[df["split"] == "test"].reset_index(drop=True)
    print(f"Train: {len(train)} emails | Val: {len(val)} | Test: {len(test)}")
    print("\nLabel counts in train:")
    print(train[LABEL_COLS].sum().rename(index=dict(zip(LABEL_COLS, LABEL_NAMES))).to_string())
    return train, val, test

train_df, val_df, test_df = load_splits(DATA_PATH)

Using device: cuda
Train: 187 emails | Val: 37 | Test: 37

Label counts in train:
Request         81
Propose         58
Commit          38
Meeting         54
Subjective     136
Informative     22


## 2. Tokenization and Dataset

BERT cannot read raw text. Each email must go through two steps before the model can use it:

1. **Tokenization**: the tokenizer converts each word (or sub-word) into an integer ID from BERT's vocabulary. It also adds two special tokens automatically: `[CLS]` at the start and `[SEP]` at the end, which BERT always requires.

2. **Padding and attention mask**: emails have different lengths, but the model requires all inputs to be the same size (`MAX_LEN`). Short emails are padded with zeros. The attention mask is a list of 1s (real tokens) and 0s (padding) that tells BERT which positions to ignore.

`EmailDataset` wraps the dataframe so PyTorch can retrieve one tokenized email at a time. `DataLoader` groups them into batches and shuffles the training set at each epoch.

In [16]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# Load the pre-trained BERT tokenizer — no fitting needed, it already knows the vocabulary
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class EmailDataset(Dataset):
    def __init__(self, df):
        self.texts  = df["body"].fillna("").tolist()
        self.labels = df[LABEL_COLS].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",   # pad short emails to MAX_LEN
            truncation=True,        # cut emails longer than MAX_LEN
            return_tensors="pt",    # return PyTorch tensors directly
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),       # token IDs
            "attention_mask": encoding["attention_mask"].squeeze(0),  # 1=real, 0=padding
            "labels":         torch.tensor(self.labels[idx]),         # 6 binary labels
        }


train_dataset = EmailDataset(train_df)
val_dataset   = EmailDataset(val_df)
test_dataset  = EmailDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# Sanity check: inspect one batch
batch = next(iter(train_loader))
print("input_ids shape:     ", batch["input_ids"].shape)       # expect (8, 256)
print("attention_mask shape:", batch["attention_mask"].shape)  # expect (8, 256)
print("labels shape:        ", batch["labels"].shape)          # expect (8, 6)
print("\nFirst email labels (6 binary values):", batch["labels"][0].tolist())

input_ids shape:      torch.Size([8, 256])
attention_mask shape: torch.Size([8, 256])
labels shape:         torch.Size([8, 6])

First email labels (6 binary values): [0.0, 1.0, 1.0, 1.0, 1.0, 0.0]


## 3. Model — BERT + LoRA

This block has two parts:

**Part A — Load BERT with a classification head**: `AutoModelForSequenceClassification` takes the pre-trained BERT base and adds a linear layer on top with 6 outputs (one per label). Setting `problem_type="multi_label_classification"` tells HuggingFace to use sigmoid + BCE loss internally during training.

**Part B — Apply LoRA**: all BERT weights are frozen. LoRA inserts small trainable matrices inside the attention layers (`query` and `value` projections). Key parameters:
- `r=8`: the rank of the LoRA matrices. Controls how many parameters are added. 8 is a standard conservative value.
- `lora_alpha=16`: a scaling factor that controls how much influence the LoRA updates have relative to the frozen weights.
- `lora_dropout=0.1`: dropout applied to LoRA layers to reduce overfitting.

The parameter count printed at the end confirms LoRA is working — only ~2% of parameters should be trainable.

In [20]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

# --- Part A: Load BERT with classification head ------------------------------
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",  # sigmoid + BCE loss
)

# --- Part B: Apply LoRA ------------------------------------------------------
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,       # sequence classification task
    r=8,                              # rank of the LoRA matrices
    lora_alpha=16,                    # scaling factor for LoRA updates
    lora_dropout=0.1,                 # dropout inside LoRA layers
    target_modules=["query", "key", "value", "dense"], # which attention matrices to adapt
    bias="none",                      # do not train bias terms
)

model = get_peft_model(base_model, lora_config)
model = model.to(device)

# --- Confirm trainable parameter count ---------------------------------------
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} of {total:,} total ({100 * trainable / total:.2f}%)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 1,344,006 of 110,830,860 total (1.21%)


## 4. Training loop

Three components work together here:

- **AdamW optimizer**: updates LoRA parameters after each batch, adapting the step size individually per parameter — standard choice for transformer fine-tuning.
- **Linear scheduler with warmup**: the learning rate grows gradually during the first few steps (warmup) to avoid large unstable updates at the start, then decreases linearly to zero by the end of training.
- **Epoch loop**: each epoch is one full pass over the training set. At the end of each epoch, the model is evaluated on the validation set. The checkpoint with the best validation Micro F1 is saved — this prevents keeping a model that overfits to training data.

In [21]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
import copy

# --- Class-weighted BCE loss -------------------------------------------------
y_train_np = train_df[LABEL_COLS].values
pos        = y_train_np.sum(axis=0)
neg        = len(y_train_np) - pos
pos_weight = torch.tensor(neg / np.maximum(pos, 1), dtype=torch.float32).to(device)
criterion  = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# --- Optimizer ---------------------------------------------------------------
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),  # only LoRA params
    lr=LR,
    weight_decay=0.01,
)

# --- Scheduler ---------------------------------------------------------------
total_steps  = len(train_loader) * EPOCHS
warmup_steps = total_steps // 10  # warm up for the first 10% of steps

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# --- Helper: evaluate on a dataloader ----------------------------------------
def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            total_loss += criterion(outputs.logits, labels).item()

            probs = torch.sigmoid(outputs.logits)          # convert logits to probabilities
            preds = (probs >= THRESHOLD).int().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    micro_f1   = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    avg_loss   = total_loss / len(loader)
    return avg_loss, micro_f1

# --- Training loop -----------------------------------------------------------
best_val_f1  = 0.0
best_weights = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)          # BCE loss computed internally by HuggingFace
        loss.backward()                 # backpropagation

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding gradients
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    val_loss, val_f1 = evaluate(val_loader)

    print(f"Epoch {epoch}/{EPOCHS} — train loss: {train_loss:.4f} | val loss: {val_loss:.4f} | val Micro F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_weights = copy.deepcopy(model.state_dict())
        print(f"  → New best model saved (val Micro F1: {best_val_f1:.4f})")

# Restore best checkpoint before evaluation
model.load_state_dict(best_weights)
print(f"\nTraining complete. Best val Micro F1: {best_val_f1:.4f}")

Epoch 1/5 — train loss: 0.9206 | val loss: 1.0000 | val Micro F1: 0.5419
  → New best model saved (val Micro F1: 0.5419)
Epoch 2/5 — train loss: 0.8749 | val loss: 1.0583 | val Micro F1: 0.4516
Epoch 3/5 — train loss: 0.8018 | val loss: 0.9869 | val Micro F1: 0.5738
  → New best model saved (val Micro F1: 0.5738)
Epoch 4/5 — train loss: 0.7450 | val loss: 0.9476 | val Micro F1: 0.6473
  → New best model saved (val Micro F1: 0.6473)
Epoch 5/5 — train loss: 0.6890 | val loss: 0.9116 | val Micro F1: 0.6840
  → New best model saved (val Micro F1: 0.6840)

Training complete. Best val Micro F1: 0.6840


## 5. Evaluation and saving predictions

This block evaluates the best model on the test set using the same metrics as the TF-IDF baseline:
Micro F1, Macro F1, Hamming loss, and a per-label breakdown.

Predictions are saved to `results/bc3_bert_lora_test_predictions.csv` with exactly the same
column schema as `bc3_baseline_test_predictions.csv`, so both files can be loaded and compared
directly in a single dataframe.

In [22]:
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, hamming_loss

# --- Full evaluation on test set ---------------------------------------------
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.sigmoid(outputs.logits)
        preds   = (probs >= THRESHOLD).int().cpu().numpy()

        all_preds.append(preds)
        all_labels.append(batch["labels"].numpy())

y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

# --- Metrics -----------------------------------------------------------------
print("=== TEST ===")
print(f"Micro F1:        {f1_score(y_true, y_pred, average='micro', zero_division=0):.3f}")
print(f"Macro F1:        {f1_score(y_true, y_pred, average='macro', zero_division=0):.3f}")
print(f"Micro Precision: {precision_score(y_true, y_pred, average='micro', zero_division=0):.3f}")
print(f"Micro Recall:    {recall_score(y_true, y_pred, average='micro', zero_division=0):.3f}")
print(f"Hamming loss:    {hamming_loss(y_true, y_pred):.3f}")
print()
print("Per-label breakdown:")
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, zero_division=0))

# --- Save predictions --------------------------------------------------------
pred_df = test_df[["listno", "email_num"]].copy()
for i, name in enumerate(LABEL_NAMES):
    pred_df[f"pred_{name}"] = y_pred[:, i]
    pred_df[f"true_{name}"] = y_true[:, i]

os.makedirs(RESULTS_DIR, exist_ok=True)
out_path = os.path.join(RESULTS_DIR, "bc3_bert_lora_test_predictions.csv")
pred_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

=== TEST ===
Micro F1:        0.637
Macro F1:        0.525
Micro Precision: 0.522
Micro Recall:    0.818
Hamming loss:    0.369

Per-label breakdown:
              precision    recall  f1-score   support

     Request       0.52      0.72      0.60        18
     Propose       0.40      0.71      0.51        14
      Commit       0.46      0.86      0.60        14
     Meeting       0.48      0.79      0.59        14
  Subjective       0.76      0.93      0.84        28
 Informative       0.00      0.00      0.00         0

   micro avg       0.52      0.82      0.64        88
   macro avg       0.44      0.67      0.53        88
weighted avg       0.56      0.82      0.66        88
 samples avg       0.55      0.84      0.60        88

Saved: ../results/bc3_bert_lora_test_predictions.csv
